In [1]:


import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

# =========================================================
# 1. LOAD DATASET
# =========================================================

data = load_iris()

X = data.data
y = data.target

# Split data into training and testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Normalize the input data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# =========================================================
# 2. PSO PARAMETERS
# =========================================================

num_particles = 10
num_iterations = 10

w = 0.7       # Inertia weight
c1 = 1.5      # Cognitive coefficient
c2 = 1.5      # Social coefficient

min_neurons = 5
max_neurons = 50


# =========================================================
# 3. INITIALIZE PARTICLES
# =========================================================

# Each particle represents:
# [neurons in hidden layer 1, neurons in hidden layer 2]

positions = np.random.randint(
    min_neurons,
    max_neurons + 1,
    size=(num_particles, 2)
)

# Initialize velocities
velocities = np.random.uniform(
    -2,
    2,
    size=(num_particles, 2)
)


# =========================================================
# 4. FITNESS FUNCTION
# =========================================================

def fitness(position):

    # Number of neurons in hidden layers
    neurons1 = int(round(position[0]))
    neurons2 = int(round(position[1]))

    # Keep values within the allowed range
    neurons1 = np.clip(neurons1, min_neurons, max_neurons)
    neurons2 = np.clip(neurons2, min_neurons, max_neurons)

    # Create DNN
    model = MLPClassifier(
        hidden_layer_sizes=(neurons1, neurons2),
        activation='relu',
        solver='adam',
        max_iter=500,
        random_state=42
    )

    # Train DNN
    model.fit(X_train, y_train)

    # Predict test data
    prediction = model.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, prediction)

    # PSO minimizes error
    error = 1 - accuracy

    return error


# =========================================================
# 5. INITIALIZE PERSONAL BEST
# =========================================================

pbest_positions = positions.copy()

pbest_scores = np.zeros(num_particles)

for i in range(num_particles):

    pbest_scores[i] = fitness(positions[i])


# =========================================================
# 6. INITIALIZE GLOBAL BEST
# =========================================================

best_index = np.argmin(pbest_scores)

gbest_position = pbest_positions[best_index].copy()

gbest_score = pbest_scores[best_index]


# =========================================================
# 7. PSO OPTIMIZATION
# =========================================================

for iteration in range(num_iterations):

    print("\nIteration:", iteration + 1)

    for i in range(num_particles):

        # Random numbers
        r1 = np.random.random(2)
        r2 = np.random.random(2)

        # -------------------------------------------------
        # UPDATE VELOCITY
        # -------------------------------------------------

        velocities[i] = (
            w * velocities[i]
            + c1 * r1 * (pbest_positions[i] - positions[i])
            + c2 * r2 * (gbest_position - positions[i])
        )

        # -------------------------------------------------
        # UPDATE POSITION
        # -------------------------------------------------

        positions[i] = positions[i] + velocities[i]

        # Keep neurons within range
        positions[i] = np.clip(
            positions[i],
            min_neurons,
            max_neurons
        )

        # -------------------------------------------------
        # CALCULATE FITNESS
        # -------------------------------------------------

        current_score = fitness(positions[i])

        # -------------------------------------------------
        # UPDATE PERSONAL BEST
        # -------------------------------------------------

        if current_score < pbest_scores[i]:

            pbest_scores[i] = current_score

            pbest_positions[i] = positions[i].copy()

        # -------------------------------------------------
        # UPDATE GLOBAL BEST
        # -------------------------------------------------

        if current_score < gbest_score:

            gbest_score = current_score

            gbest_position = positions[i].copy()

    # Display best result of current iteration

    print(
        "Best Hidden Layer 1:",
        int(round(gbest_position[0]))
    )

    print(
        "Best Hidden Layer 2:",
        int(round(gbest_position[1]))
    )

    print(
        "Best Error:",
        gbest_score
    )

    print(
        "Best Accuracy:",
        (1 - gbest_score) * 100,
        "%"
    )


# =========================================================
# 8. CREATE FINAL DNN
# =========================================================

best_neurons1 = int(round(gbest_position[0]))
best_neurons2 = int(round(gbest_position[1]))

final_model = MLPClassifier(
    hidden_layer_sizes=(best_neurons1, best_neurons2),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)


# =========================================================
# 9. TRAIN FINAL DNN
# =========================================================

final_model.fit(X_train, y_train)


# =========================================================
# 10. TEST FINAL DNN
# =========================================================

final_prediction = final_model.predict(X_test)

final_accuracy = accuracy_score(
    y_test,
    final_prediction
)


# =========================================================
# 11. DISPLAY FINAL RESULT
# =========================================================

print("\n========================================")
print("          FINAL PSO-DNN RESULT")
print("========================================")

print(
    "Optimal Hidden Layer 1:",
    best_neurons1
)

print(
    "Optimal Hidden Layer 2:",
    best_neurons2
)

print(
    "Minimum Error:",
    gbest_score
)

print(
    "Final Accuracy:",
    final_accuracy * 100,
    "%"
)

print("========================================")


#VARALAKSHMI SR [1BM25CS537]

/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(



Iteration: 1


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Best Hidden Layer 1: 33
Best Hidden Layer 2: 42
Best Error: 0.033333333333333326
Best Accuracy: 96.66666666666667 %

Iteration: 2
Best Hidden Layer 1: 33
Best Hidden Layer 2: 42
Best Error: 0.033333333333333326
Best Accuracy: 96.66666666666667 %

Iteration: 3
Best Hidden Layer 1: 33
Best Hidden Layer 2: 42
Best Error: 0.033333333333333326
Best Accuracy: 96.66666666666667 %

Iteration: 4


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


...

Best Hidden Layer 1: 33
Best Hidden Layer 2: 42
Best Error: 0.033333333333333326
Best Accuracy: 96.66666666666667 %

Iteration: 7
Best Hidden Layer 1: 26
Best Hidden Layer 2: 39
Best Error: 0.0
Best Accuracy: 100.0 %

Iteration: 8
Best Hidden Layer 1: 26
Best Hidden Layer 2: 39
Best Error: 0.0
Best Accuracy: 100.0 %

Iteration: 9


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Best Hidden Layer 1: 26
Best Hidden Layer 2: 39
Best Error: 0.0
Best Accuracy: 100.0 %

Iteration: 10


/usr/local/lib/python3.13/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Best Hidden Layer 1: 26
Best Hidden Layer 2: 39
Best Error: 0.0
Best Accuracy: 100.0 %

          FINAL PSO-DNN RESULT
Optimal Hidden Layer 1: 26
Optimal Hidden Layer 2: 39
Minimum Error: 0.0
Final Accuracy: 100.0 %
